#  LangChain의 RAG 콤포넌트 - 문서 임베딩(Embeddings) 

### **학습 목표:**  임베딩 모델과 벡터 데이터베이스를 효과적으로 연동할 수 있다

### **실습 자료**: 

- data/transformer.pdf

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob  

from pprint import pprint  
import json

`(3) 문서 로드`

In [3]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

/var/folders/0s/szhm38ld0pgfxjpnvwq0hjx80000gn/T/ipykernel_32494/3020463323.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/shu/Desktop/Source Code/modulab-ai-llm-service-dev-7th/faq_bot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF 문서 개수: 15


`(4) 텍스트 분할`

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 초기화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,             # 청크 크기  
    chunk_overlap=200,           # 청크 중 중복되는 부분 크기
    length_function=len,         # 글자 수를 기준으로 분할
    separators=["\n\n", "\n", " ", ""],  # 구분자 - 재귀적으로 순차적으로 적용 
)

# PDF 문서를 텍스트로 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 52
각 청크의 길이: [986, 910, 975, 452, 933, 995, 902, 907, 996, 385, 924, 954, 216, 924, 901, 950, 995, 913, 908, 870, 945, 973, 946, 997, 196, 980, 980, 946, 938, 999, 943, 920, 734, 958, 946, 945, 617, 983, 988, 994, 624, 944, 909, 941, 914, 986, 925, 927, 847, 812, 815, 818]


# 문서 임베딩(Document Embedding)

- 개념: 
    - 텍스트를 벡터(숫자 배열)로 변환하는 과정
    - 문서의 의미적 특성을 수치화하여 컴퓨터가 이해하고 처리할 수 있는 형태로 변환 

- 목적:
    - 텍스트 간 유사도 계산 가능
    - 벡터 데이터베이스 저장 및 검색
    - 의미 기반 문서 검색 구현

- LangChain의 임베딩 모델 종류:
    - OpenAI 임베딩
    - HuggingFace 임베딩 
    - Ollama 임베딩

### 1. **OpenAI**

- LangChain에서 가장 널리 사용되는 임베딩 모델 중 하나

- 주요 특징:
    1. 고품질의 임베딩 생성
    2. 다양한 언어 지원 (다국어 지원)
    3. 일관된 성능
    4. 손쉬운 통합

- 사용시 주의사항:
    1. API 키 설정이 필요 (환경 변수 OPENAI_API_KEY)
    2. API 사용량에 따른 비용 발생
    3. 긴 텍스트는 자동으로 분할되지 않으므로 필요시 TextSplitter를 사용


- 모델별 특징

    | 모델명 | 가격 효율 (페이지/1달러) | 성능 (MTEB) | 최대 입력 | 기본 차원 | 특징 및 추천 |
    | :--- | :---: | :---: | :---: | :---: | :--- |
    | **`text-embedding-3-small`** | **62,500** | 62.3% | 8,191 | 1,536 | **[가성비]** ada-002 대비 5배 저렴하고 성능 우수. 일반적인 RAG 구축 시 1순위. |
    | **`text-embedding-3-large`** | 9,615 | **64.6%** | 8,191 | 3,072 | **[고성능]** 미세한 의미 차이 구분이 중요할 때 사용. 차원 축소 기능 지원. |
    | **`text-embedding-ada-002`** | 12,500 | 61.0% | 8,191 | 1,536 | **[레거시]** 기존에 널리 쓰이던 모델이나, 현재는 v3-small 사용을 권장함. |

> **💡 Tip:** `text-embedding-3` 계열은 **Matryoshka Embedding** 기술이 적용되어, 저장 공간 절약을 위해 임베딩 차원(예: 1536 → 512)을 줄여서 요청해도 성능 하락이 매우 적습니다.

`(1) embedding 모델`

In [5]:
from langchain_openai import OpenAIEmbeddings

# OpenAIEmbeddings 모델 생성
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",  # 사용할 모델 이름
    dimensions=None, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 객체 출력
embeddings_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x13bcc4230>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x13b5ee810>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [6]:
# 임베딩 모델의 컨텍스트 길이 확인
embeddings_model.embedding_ctx_length

8191

In [7]:
# 임베딩 모델의 임베딩 차원 확인 - 기본값 (None)
embeddings_model.dimensions

In [8]:
# OpenAIEmbeddings 모델 생성할 때 임베딩 차원을 지정하는 예시
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름
    dimensions=512, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 모델의 임베딩 차원 확인 
embeddings_model.dimensions

512

In [9]:
# OpenAIEmbeddings 모델 생성
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# 임베딩 객체 출력
embeddings_openai

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x14f343fb0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x14f53a0c0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

`(2) embed_documents 사용`

In [10]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_openai = embeddings_openai.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_openai)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_openai[0])}")
print(document_embeddings_openai[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1536
[-0.0022602081298828125, 0.01212310791015625, -0.0024471282958984375, 0.01502227783203125, 0.0184478759765625, -0.045684814453125, -0.0031032562255859375, 0.046966552734375, -0.01824951171875, -0.031829833984375, 0.006900787353515625, 0.00699615478515625, -0.018585205078125, -0.027099609375, 0.00498199462890625, -0.015594482421875, -0.041656494140625, 0.01003265380859375, 0.05157470703125, -0.045806884765625, -0.0146331787109375, -0.027801513671875, -0.018768310546875, -0.0227203369140625, 0.004375457763671875, -0.03973388671875, 0.05023193359375, 0.0177001953125, 0.0003368854522705078, -0.02337646484375, 0.0489501953125, -0.01526641845703125, -0.02996826171875, -0.061676025390625, 0.020538330078125, 0.03546142578125, 0.0025959014892578125, -0.006885528564453125, -0.005100250244140625, 0.0247344970703125, 0.007564544677734375, 0.0275115966796875, -0.0025386810302734375, 0.0260772705078125, -0.0179443359375, 0.005462646484375, -0.0258331298828125, 0.003633

`(3) embed_query 사용`

In [11]:
embedded_query_openai = embeddings_openai.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query_openai)}")
print(embedded_query_openai)

쿼리 임베딩 벡터의 차원: 1536
[-0.0224456787109375, 0.0222015380859375, 0.0003859996795654297, 0.00574493408203125, 0.0122528076171875, -0.044769287109375, -0.026275634765625, 0.035797119140625, -0.0025959014892578125, 0.01476287841796875, -0.0019893646240234375, 0.0009360313415527344, -0.004917144775390625, -0.0736083984375, 0.00899505615234375, -0.0154266357421875, -0.05987548828125, -0.02239990234375, 0.022491455078125, -0.06915283203125, -0.02923583984375, 0.023223876953125, -0.036865234375, 0.0041046142578125, 0.011077880859375, -0.052734375, 0.01071929931640625, 9.28044319152832e-05, -0.0108489990234375, -0.03424072265625, 0.0253143310546875, -0.018829345703125, -0.0015420913696289062, -0.058807373046875, 0.0498046875, -0.0034999847412109375, -0.0028629302978515625, -0.00855255126953125, -0.00460052490234375, 0.0228729248046875, -0.01197052001953125, 0.0379638671875, 0.0033168792724609375, 0.035400390625, -0.051361083984375, 0.034393310546875, -0.0285491943359375, 0.004730224609375, -0.013

`(4) 유사도 기반 검색`

In [12]:
from langchain_community.utils.math import cosine_similarity
import numpy as np

# 쿼리와 가장 유사한 문서 찾기 함수
def find_most_similar(
        query: str, 
        doc_embeddings: np.ndarray,
        embeddings_model  # 기본값 제거, 명시적 전달 강제
        ) -> tuple[str, float]:
    """
    쿼리와 가장 유사한 문서를 찾는 함수
    
    Args:
        query: 검색 쿼리 문자열
        doc_embeddings: 문서 임베딩 배열
        embeddings_model: 임베딩 모델 객체
    
    Returns:
        tuple: (가장 유사한 문서, 유사도 점수)
    """
    # 쿼리 임베딩: OpenAI 임베딩 사용 
    query_embedding = embeddings_model.embed_query(query)

    # 코사인 유사도 계산
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

    # 가장 유사한 문서 인덱스 찾기
    most_similar_idx = np.argmax(similarities)

    # 가장 유사한 문서와 유사도 반환: 문서, 유사도
    return documents[most_similar_idx], similarities[most_similar_idx]

# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(
        query, 
        document_embeddings_openai, 
        embeddings_model=embeddings_openai
        )
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()
    

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7114

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.6826

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.7053



### 2. **Huggingface**

- LangChain에서 오픈소스 기반의 대표적인 임베딩 모델

- 주요 특징:
    1. 로컬 환경에서 실행 가능
    2. 다양한 사전학습 모델 지원
    3. 커스텀 모델 학습 및 적용 가능
    4. 무료 사용 가능 (API 비용 없음)

- 사용시 주의사항:
    1. 로컬 컴퓨팅 자원 필요 (CPU/GPU)
    2. 초기 모델 다운로드 시간 소요
    3. 메모리 사용량 고려 필요
    4. transformers 라이브러리 설치 필요

- 임베딩 벡터 특성:
    1. 모델별로 다양한 차원 제공 (128 ~ 1024)
    2. sentence-transformers 기반 구현
    3. BERT 계열 모델 구조 사용
    4. 코사인 유사도 기반 검색 최적화

- 대표적인 임베딩 모델:

    | 모델명 (Hugging Face ID) | 차원 | 언어 | 특징 및 추천 용도 |
    | :--- | :---: | :---: | :--- |
    | **`all-MiniLM-L6-v2`** | 384 | **영어** | **[영어 표준/경량]** 매우 빠르고 메모리 효율이 좋음. 영어 전용 검색/분류 작업의 입문용 모델. |
    | **`all-mpnet-base-v2`** | 768 | **영어** | **[영어 고성능]** MiniLM보다 느리지만, 문장의 뉘앙스를 가장 정확하게 포착함. 영어권 RAG의 표준. |
    | **`paraphrase-multilingual-MiniLM-L12-v2`** | 384 | 다국어 | **[다국어 경량]** 한국어를 포함한 50개국어 지원. 속도가 빨라 실시간 서비스에 적합. |
    | **`intfloat/multilingual-e5-large`** | 1024 | 다국어 | **[다국어 고성능]** 다국어 벤치마크 상위권 모델. (사용 시 `query:`, `passage:` 접두어 필요) |
    | **`BAAI/bge-m3`** | 1024 | 다국어 | **[한국어 최적]** 한국어 처리 성능이 매우 뛰어나며, 긴 문장(8192 토큰)도 처리 가능. |


`(1) embedding 모델`

- langchain_huggingface 설치 필요

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
embeddings_gemma = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",          # 사용할 모델 이름 - 구글의 경량 임베딩 모델
    # model_kwargs={'device': 'cuda'}  # GPU 사용시
    model_kwargs={'device': 'mps'}     # Mac M1 사용시
)

# 임베딩 객체 출력
embeddings_gemma

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 70099.29it/s]


HuggingFaceEmbeddings(model_name='BAAI/bge-m3', cache_folder=None, model_kwargs={'device': 'mps'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

`(2) embed_documents 사용`

In [16]:
# 문서 임베딩
document_embeddings_gemma = embeddings_gemma.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_gemma)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_gemma[0])}")
print(document_embeddings_gemma[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1024
[-0.039414484053850174, 0.008764827623963356, -0.012681616470217705, 0.0024531735107302666, -0.008944731205701828, -0.0073836552910506725, -0.005377382505685091, -0.009055779315531254, 0.03291524201631546, 0.006045561749488115, -0.02701297216117382, -0.02774093858897686, 0.0004441519267857075, 0.030136574059724808, 0.0172429196536541, 0.01709037832915783, 0.025524871423840523, -0.02185601182281971, -0.011341308243572712, -0.0570225864648819, -0.00030165869975462556, 0.01354304514825344, -0.007450129371136427, 0.018574519082903862, 0.0028946849051862955, 0.008630719035863876, -0.0007445486844517291, -0.028904221951961517, 0.020727766677737236, -0.020500702783465385, 0.008069789968430996, -0.026754241436719894, 0.003963078372180462, -0.016303904354572296, -0.07406214624643326, -0.03365033492445946, -0.023871438577771187, -0.03455006703734398, -0.03478589653968811, 0.0054830643348395824, -0.050033584237098694, -0.00280371424742043, -0.023147009313106537, -0.

`(3) embed_query 사용`

In [17]:
embedded_query = embeddings_gemma.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 1024
[-0.037039026618003845, -0.0048380037769675255, 0.00293731689453125, -0.015514614060521126, -0.000944195780903101, -0.04150165244936943, -0.00657447287812829, 0.011289631947875023, 0.02161405049264431, 0.004928720649331808, -0.02034064754843712, 0.016905182972550392, -0.012874133884906769, 0.0055189174599945545, 0.014988369308412075, 0.024228811264038086, 0.007369156461209059, -0.028049813583493233, -0.014939038082957268, -0.05185193195939064, -0.006705006118863821, -0.009251507930457592, -0.016980893909931183, 0.006491500418633223, 0.0529317744076252, 0.04813729226589203, -0.008069595322012901, -0.023171765729784966, 0.018143028020858765, -0.011328169144690037, -0.00424041086807847, -0.006354685872793198, -0.002271785866469145, 0.014329486526548862, -0.03563671559095383, -0.008155844174325466, -0.011798140592873096, -0.045424021780490875, -0.04073285311460495, 0.0022138948552310467, -0.012132338248193264, 0.017896011471748352, -0.019144687801599503, -0.041924424469

`(4) 유사도 기반 검색`

In [18]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_gemma, embeddings_model=embeddings_gemma) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7269

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.7057

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6843



### 3. **Ollama (로컬 실행 최적화)**

LangChain에서 로컬 LLM 및 임베딩 모델을 가장 손쉽게 실행할 수 있는 플랫폼입니다. 외부 API 전송 없이 로컬 자원(GPU/CPU)만 사용하므로 데이터 보안과 비용 절감에 최적화되어 있습니다.

**주요 특징:**
1. **완전한 로컬 실행:** 데이터가 외부로 유출되지 않아 기업 내부(On-premise) 구축에 적합.
2. **빠른 추론 속도:** C++ 기반의 런타임과 양자화(Quantization) 기술로 최적화됨.
3. **간편한 배포:** Docker 기반으로 모델 설치 및 실행이 매우 간단함 (`ollama pull 모델명`).

**사용 시 주의사항:**
1. **서버 실행 필수:** 백그라운드에서 `ollama serve`가 실행 중이어야 함.
2. **리소스 관리:** 고성능 모델(Large) 사용 시 충분한 RAM/VRAM 필요.
3. **API 설정:** LangChain 등에서 호출 시 엔드포인트(`localhost:11434`) 확인 필요.

**대표적인 임베딩 모델 비교:**

| 모델명 (Model Tag) | 차원 | 언어 | 특징 및 추천 용도 |
| :--- | :---: | :---: | :--- |
| **`nomic-embed-text`** | 768 | 영어 | **[Ollama 표준]** 긴 문맥(8192 토큰)을 지원하며, 오픈소스 중 밸런스가 가장 우수함. |
| **`mxbai-embed-large`** | 1024 | 영어 | **[SOTA 성능]** MTEB 리더보드 상위권 모델. 검색 정확도가 매우 높음. |
| **`snowflake-arctic-embed`** | 1024 | 영어 | **[검색 최적화]** Snowflake사가 RAG 및 대규모 검색 작업에 특화하여 설계함. |
| **`bge-m3`** | 1024 | **다국어** | **[한국어 추천]** Ollama에서 사용 가능한 가장 강력한 다국어/한국어 모델. |
| **`all-minilm`** | 384 | 영어 | **[초경량]** 속도가 매우 빠르고 CPU 환경에서도 부담 없이 실행 가능. |

**임베딩 벡터 특성:**
1. **고정 차원:** 모델별로 384~1024의 고정된 벡터 차원을 가짐.
2. **자동 양자화:** 원본 모델을 4비트(Q4_0) 등으로 압축하여 메모리 사용량을 대폭 줄임.

`(1) embedding 모델`

- langchain_ollama 설치 필요

In [19]:
from langchain_ollama import OllamaEmbeddings 

# OllamaEmbeddings 모델 생성
# embeddings_ollama = OllamaEmbeddings(
#     model="nomic-embed-text",          # 사용할 모델 이름
#     base_url="http://localhost:11434"  # Ollama 서버 주소
# )
embeddings_ollama = OllamaEmbeddings(model="bge-m3")

# 임베딩 객체 출력
embeddings_ollama

OllamaEmbeddings(model='bge-m3', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

`(2) embed_documents 사용`

In [20]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_ollama = embeddings_ollama.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_ollama)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_ollama[0])}")
print(document_embeddings_ollama[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1024
[-0.03934616, 0.008725222, -0.01283989, 0.002369647, -0.009001697, -0.0073944954, -0.005533404, -0.009016167, 0.032826923, 0.0059871487, -0.0272862, -0.02791971, 0.00036138474, 0.030159809, 0.017178033, 0.01723762, 0.025453616, -0.021885864, -0.011496796, -0.056941666, -0.00024311537, 0.013517512, -0.007446473, 0.018559905, 0.002889891, 0.008624155, -0.0008203585, -0.02906104, 0.020742835, -0.020622645, 0.008179191, -0.02685354, 0.0036557212, -0.016159842, -0.07388179, -0.03372631, -0.023856977, -0.03434238, -0.03461791, 0.0055033867, -0.05011897, -0.0029549194, -0.023078613, -0.075004704, -0.011301758, -0.028950157, -0.034177683, -0.025457088, -0.061371237, 0.013589068, 0.024477227, -0.03144307, 0.050583024, 0.012999215, -0.063970834, 0.027330725, 0.0070550027, 0.01038709, -0.06404145, -0.0161907, -0.02071215, 0.030176383, -0.0063339323, -0.013089502, -0.0006902965, 0.046951998, 0.007844013, 0.02963074, -0.03480933, -0.036349505, 0.021500956, 0.01768091,

`(3) embed_query 사용`

In [21]:
embedded_query = embeddings_ollama.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 1024
[-0.036800895, -0.004901595, 0.0028816732, -0.015532954, -0.0009126782, -0.04167636, -0.0067102746, 0.011346931, 0.021551177, 0.004906359, -0.020482162, 0.01679398, -0.012878139, 0.0054356814, 0.014995769, 0.024306148, 0.0074279984, -0.027959641, -0.015135979, -0.05182668, -0.006607204, -0.0091784885, -0.017132942, 0.0065896492, 0.05290853, 0.048060328, -0.008178606, -0.023206752, 0.018088752, -0.01138985, -0.0043378626, -0.006354323, -0.0024372109, 0.014357538, -0.035445187, -0.008221224, -0.011810933, -0.04529177, -0.040649865, 0.002170825, -0.01229813, 0.017701318, -0.019154567, -0.04193543, 0.0009956096, -0.03901822, -0.024441313, -0.024550384, -0.02219327, -0.004478447, 0.031530853, -0.04875265, 0.018001532, 0.02526104, 0.002390402, 0.048537012, -0.008122446, 0.028102865, -0.073755965, -0.020699373, -0.026367156, -0.00749526, -0.038773578, -0.0173926, 0.019238278, 0.06373703, 0.020386573, -0.011859377, -0.018533485, -0.040780757, 0.0025353292, 0.04148482, -0.05

`(4) 유사도 기반 검색`

In [22]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_ollama, embeddings_model=embeddings_ollama) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7271

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.7051

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6837



# [실습 프로젝트]

1. OpenAI text-embedding-3-small 임베딩 모델을 초기화합니다. 
2. 임베딩 차원을 각각 512와 1536으로 구분하여 2개의 모델을 생성합니다. 
3. 아래 주어진 문장들의 임베딩을 생성합니다. (임베딩 차원이 512, 1536인 경우 각각 2개씩 생성)
   - 문장1: "인공지능은 현대 사회를 변화시키고 있다"
   - 문장2: "AI 기술이 우리의 미래를 바꾸고 있다"
4. 생성된 임베딩의 차원을 출력합니다. (임베딩 차원이 512, 1536인 경우를 각각 출력)
5. 두 문장 간의 코사인 유사도를 계산합니다. (임베딩 차원이 512, 1536인 경우를 각각 비교)
6. [추가] 임베딩 차원(512 vs 1536)에 따른 유사도 차이를 분석하고, 어떤 차원이 더 효과적인지 생각해보세요.

In [ ]:
# 여기에 코드를 작성하세요.